# 02 -- Hyperparameter Optimization: EuroBERT-210m

Optuna-based hyperparameter tuning with 3-fold stratified cross-validation.
Supports chunked execution: can be run multiple times, resuming from where it left off.

**Run modes:**
- `RUN_MODE = "new"` -- Start fresh study with new database
- `RUN_MODE = "continue"` -- Resume existing study, add more trials

**Prerequisites:** GPU runtime (T4 / L4), `HF_TOKEN` in Colab Secrets.

## Run Configuration
**Adjust before each run!**

In [ ]:
# ============================================================
# CHUNK-RUN CONFIGURATION — ADJUST BEFORE EACH RUN
# ============================================================

# Mode: "new"      = create a new study / database
#        "continue" = resume an existing study
RUN_MODE = "new"

# Number of trials to run in THIS session (regardless of complete/pruned/failed)
N_TRIALS_THIS_RUN = 5

# DB path (only relevant for "continue" mode):
#   - None  = automatically use the most recently created DB
#   - path  = specify explicitly, e.g. "/content/drive/MyDrive/thesis_reports/euroBERT_210m/optuna_db/eurobert_210m_hpt_phase1_001.db"
DB_PATH_OVERRIDE = None

# Study name (must stay the same across all runs!)
STUDY_NAME = "eurobert_210m_hpt_phase1"

# Optional: per-run timeout in seconds (None = no limit)
OPTUNA_TIMEOUT = None

print(f"RUN_MODE:           {RUN_MODE}")
print(f"N_TRIALS_THIS_RUN:  {N_TRIALS_THIS_RUN}")
print(f"DB_PATH_OVERRIDE:   {DB_PATH_OVERRIDE}")
print(f"STUDY_NAME:         {STUDY_NAME}")

In [ ]:
# === SETUP ===
import os, sys

# Clone / update the repository
REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

# Install dependencies (including optuna, plotly, kaleido for HPT)
!pip install -q transformers[sentencepiece] datasets huggingface_hub \
    scikit-learn matplotlib seaborn tqdm pandas accelerate evaluate \
    optuna plotly kaleido

# Mount Google Drive (persistent reports + Optuna DB)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Make pipeline_utils and hpt_utils importable (both live in the same directory)
PIPELINE_DIR = f"{REPO}/Python/classification_pipeline/euroBERT_210m"
HPT_DIR = PIPELINE_DIR
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)
import hpt_utils as hu
importlib.reload(hu)

# Create all output directories on Google Drive
pu.ensure_drive_dirs()

# HuggingFace login
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

# Auto-shutdown watchdog: cap maximum runtime (safety net for Colab Pro)
import threading, time as _time
MAX_RUNTIME_HOURS = 7.5

def _auto_shutdown():
    _time.sleep(MAX_RUNTIME_HOURS * 3600)
    print(f"\n[WATCHDOG] Max runtime ({MAX_RUNTIME_HOURS}h) reached. Terminating runtime.")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass
threading.Thread(target=_auto_shutdown, daemon=True).start()

print(f"Reports directory:  {pu.REPORTS_DIR}")
print(f"Optuna DB directory: {pu.OPTUNA_DB_DIR}")
print(f"Setup complete. Watchdog: runtime will terminate after {MAX_RUNTIME_HOURS}h max.")

In [ ]:
# ===== HPT CONFIGURATION =====
import torch
import numpy as np

MODEL_ID = "EuroBERT/EuroBERT-210m"
MODEL_SHORT_NAME = "eurobert_210m"
MAX_LENGTH = 2048
RANDOM_SEED = 42

# ----- Cross-Validation -----
N_FOLDS = 3

# ----- Optuna -----
OPTUNA_SEED = 42

# ----- Fixed Training Parameters -----
FIXED_GRADIENT_CHECKPOINTING = False
FIXED_LOGGING_STEPS = 10
FIXED_REPORT_TO = "tensorboard"
FIXED_DATALOADER_NUM_WORKERS = 4
FIXED_EARLY_STOPPING_PATIENCE = 3
FIXED_GROUP_BY_LENGTH = True

# Mixed precision: adapt to available GPU
if torch.cuda.is_available():
    _gpu_cap = torch.cuda.get_device_capability()
    _gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    if _gpu_cap[0] >= 8:
        FIXED_BF16 = True
        FIXED_FP16 = False
    else:
        FIXED_BF16 = False
        FIXED_FP16 = True
    FIXED_OPTIM = "adamw_torch_fused"
    if _gpu_mem >= 40:
        FIXED_BATCH_SIZE_EVAL = 32
    elif _gpu_mem >= 20:
        FIXED_BATCH_SIZE_EVAL = 16
    else:
        FIXED_BATCH_SIZE_EVAL = 8
    print(f"GPU: {torch.cuda.get_device_name(0)} ({_gpu_mem:.1f} GB, CC {_gpu_cap[0]}.{_gpu_cap[1]})")
    print(f"  FP16={FIXED_FP16}, BF16={FIXED_BF16}, Eval Batch={FIXED_BATCH_SIZE_EVAL}")
    print(f"  Gradient Checkpointing: {FIXED_GRADIENT_CHECKPOINTING}")
else:
    raise RuntimeError("GPU required! Please switch to a GPU runtime in Colab.")

# ----- Hyperparameter Search Ranges -----
HP_RANGES = {
    "learning_rate": (1e-5, 5e-5),
    "weight_decay": (0.0, 0.1),
    "warmup_ratio": (0.0, 0.15),
    "label_smoothing_factor": (0.0, 0.1),
    "lr_scheduler_type": ["linear", "cosine"],
    "per_device_train_batch_size": [4, 8],
    "num_train_epochs": (5, 15),
}

EFFECTIVE_BATCH_SIZE = 16

# ----- Labels -----
ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

# Test split configuration
TEST_PER_CLASS = 30

print(f"\nOptuna HPT: {N_TRIALS_THIS_RUN} trials this run, {N_FOLDS}-fold CV")
print(f"Model: {MODEL_ID}")
print(f"Max length: {MAX_LENGTH}")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"RUN_MODE: {RUN_MODE}")
print(f"Labels: {len(ALL_LABELS)} classes")

In [ ]:
# ===== LOAD DATA & CREATE CUSTOM SPLIT =====
import pandas as pd
from datasets import load_dataset

np.random.seed(RANDOM_SEED)

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()
all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)

print(f"Total pool of labelled articles: {len(all_labelled)}")
print(f"Classes in dataset: {all_labelled['label'].nunique()}")
print()

# --- Test split (identical to the fine-tuning notebook) ---
test_indices = []
rest_indices = []

for label in ALL_LABELS:
    label_mask = all_labelled["label"] == label
    label_indices = all_labelled[label_mask].index.tolist()
    n_total = len(label_indices)

    if n_total < 60:
        n_test = n_total // 2
        print(f"  {label}: only {n_total} articles -> {n_test} for test (half)")
    else:
        n_test = TEST_PER_CLASS

    np.random.shuffle(label_indices)
    test_indices.extend(label_indices[:n_test])
    rest_indices.extend(label_indices[n_test:])

test_df = all_labelled.loc[test_indices].reset_index(drop=True)
cv_pool_df = all_labelled.loc[rest_indices].reset_index(drop=True)

print(f"\nTest (frozen):       {len(test_df)} articles")
print(f"CV pool (for folds): {len(cv_pool_df)} articles")

# Class distribution in CV pool
print("\nCV pool class distribution:")
cv_dist = cv_pool_df["label"].value_counts().sort_index()
for label, count in cv_dist.items():
    print(f"  {label}: {count}")
print(f"  TOTAL: {len(cv_pool_df)}")

In [ ]:
# ===== LABEL ENCODING =====
label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

cv_pool_df["label_id"] = cv_pool_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

assert cv_pool_df["label_id"].isna().sum() == 0, "Unknown labels in CV pool!"
assert test_df["label_id"].isna().sum() == 0, "Unknown labels in test set!"

print("Label mapping:")
for label, idx in label2id.items():
    print(f"  {idx:>2}: {label}")
print(f"\nNumber of classes: {len(ALL_LABELS)}")

In [ ]:
# ===== TOKENIZER + EUROBERT ROPE FIX =====
from transformers import AutoTokenizer
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Patch RoPE initialisation for EuroBERT compatibility
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init
print("ROPE_INIT_FUNCTIONS patched.")

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        truncation=True,
    )

print(f"Tokenizer loaded: {MODEL_ID}")

In [ ]:
# ===== OPTUNA OBJECTIVE WITH K-FOLD CV =====
import gc
import shutil
import optuna
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

compute_metrics = hu.make_compute_metrics(ALL_LABELS, id2label)

# Pre-compute fold indices (same folds for every trial -> fair comparison)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_indices = list(skf.split(cv_pool_df, cv_pool_df["label_id"]))

print(f"Stratified {N_FOLDS}-fold CV:")
for i, (train_idx, val_idx) in enumerate(fold_indices):
    val_labels = cv_pool_df.iloc[val_idx]["label"].value_counts()
    min_val_count = val_labels.min()
    min_val_class = val_labels.idxmin()
    print(f"  Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}, "
          f"smallest val class: {min_val_class} ({min_val_count} samples)")


def create_model(seed=42):
    """Instantiate the model with a fresh classification head."""
    torch.manual_seed(seed)

    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config.num_labels = len(ALL_LABELS)
    config.id2label = id2label
    config.label2id = label2id

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        config=config,
        ignore_mismatched_sizes=True,
        trust_remote_code=True,
    )

    import torch.nn as nn
    for name, module in model.named_modules():
        if name in ("dense", "classifier") and isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.002)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    if FIXED_GRADIENT_CHECKPOINTING and torch.cuda.is_available():
        model.gradient_checkpointing_enable()
    return model


def _cleanup_cuda():
    """Free GPU memory and run garbage collection."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


def objective(trial):
    """Optuna objective: train with sampled hyperparameters, return mean CV F1 macro."""
    # --- Sample hyperparameters ---
    lr = trial.suggest_float("learning_rate", *HP_RANGES["learning_rate"], log=True)
    wd = trial.suggest_float("weight_decay", *HP_RANGES["weight_decay"])
    warmup = trial.suggest_float("warmup_ratio", *HP_RANGES["warmup_ratio"])
    label_smooth = trial.suggest_float("label_smoothing_factor", *HP_RANGES["label_smoothing_factor"])
    scheduler = trial.suggest_categorical("lr_scheduler_type", HP_RANGES["lr_scheduler_type"])
    batch_size = trial.suggest_categorical("per_device_train_batch_size", HP_RANGES["per_device_train_batch_size"])
    epochs = trial.suggest_int("num_train_epochs", *HP_RANGES["num_train_epochs"])

    grad_accum = max(1, EFFECTIVE_BATCH_SIZE // batch_size)

    print(f"\n{'='*60}")
    print(f"Trial {trial.number}: lr={lr:.2e}, wd={wd:.3f}, warmup={warmup:.3f}, "
          f"ls={label_smooth:.3f}, sched={scheduler}, "
          f"batch={batch_size}, grad_accum={grad_accum}, epochs={epochs}")
    print(f"{'='*60}")

    fold_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
        print(f"  Fold {fold_idx + 1}/{N_FOLDS}...", end=" ", flush=True)

        _cleanup_cuda()

        fold_train_df = cv_pool_df.iloc[train_idx]
        fold_val_df = cv_pool_df.iloc[val_idx]

        train_ds = Dataset.from_pandas(
            fold_train_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
        )
        val_ds = Dataset.from_pandas(
            fold_val_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
        )

        train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
        val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
        train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
        val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

        fold_output_dir = f"/content/hpt_tmp/trial_{trial.number}_fold_{fold_idx}"
        fold_logging_dir = f"/content/hpt_tmp/tb_logs/trial_{trial.number:03d}_fold_{fold_idx}"

        metrics_callback = hu.HPTMetricsCallback(
            trial_number=trial.number,
            fold_idx=fold_idx,
            all_labels=ALL_LABELS,
            id2label=id2label,
        )

        training_args = TrainingArguments(
            output_dir=fold_output_dir,
            num_train_epochs=epochs,
            learning_rate=lr,
            weight_decay=wd,
            warmup_ratio=warmup,
            label_smoothing_factor=label_smooth,
            lr_scheduler_type=scheduler,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=FIXED_BATCH_SIZE_EVAL,
            gradient_accumulation_steps=grad_accum,
            bf16=FIXED_BF16,
            fp16=FIXED_FP16,
            gradient_checkpointing=FIXED_GRADIENT_CHECKPOINTING,
            optim=FIXED_OPTIM,
            group_by_length=FIXED_GROUP_BY_LENGTH,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_strategy="steps",
            logging_steps=FIXED_LOGGING_STEPS,
            logging_dir=fold_logging_dir,
            report_to=FIXED_REPORT_TO,
            seed=RANDOM_SEED + fold_idx,
            dataloader_num_workers=FIXED_DATALOADER_NUM_WORKERS,
            dataloader_pin_memory=True,
            disable_tqdm=False,
        )

        try:
            model = create_model(seed=RANDOM_SEED + trial.number * 100 + fold_idx)

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                data_collator=data_collator,
                compute_metrics=compute_metrics,
                callbacks=[
                    EarlyStoppingCallback(early_stopping_patience=FIXED_EARLY_STOPPING_PATIENCE),
                    metrics_callback,
                ],
            )

            trainer.train()

            # Check for NaN or learning-rate collapse detected during training
            if metrics_callback.nan_detected or metrics_callback.lr_zero_detected:
                reason = "NaN" if metrics_callback.nan_detected else "LR=0"
                print(f"{reason} detected — pruning Trial {trial.number}")
                hu.store_fold_metrics_partial(trial, fold_idx, metrics_callback, ALL_LABELS)
                trial.set_user_attr("nan_trial", True)
                trial.set_user_attr("nan_reason", reason)
                trial.set_user_attr("nan_fold_idx", fold_idx)
                raise optuna.TrialPruned()

            eval_result = trainer.evaluate()
            fold_f1 = eval_result["eval_f1_macro"]

            # Check for NaN or collapse in final evaluation
            if np.isnan(eval_result.get("eval_loss", 0)) or fold_f1 < 0.05:
                print(f"NaN/collapse in final eval (F1={fold_f1:.4f}) — pruning")
                hu.store_fold_metrics(trial, fold_idx, metrics_callback, eval_result, ALL_LABELS, id2label)
                trial.set_user_attr("nan_trial", True)
                trial.set_user_attr("nan_reason", "eval_nan")
                trial.set_user_attr("nan_fold_idx", fold_idx)
                raise optuna.TrialPruned()

            hu.store_fold_metrics(trial, fold_idx, metrics_callback, eval_result, ALL_LABELS, id2label)

            fold_scores.append(fold_f1)
            print(f"F1 Macro = {fold_f1:.4f}")

        except torch.cuda.OutOfMemoryError:
            print(f"\n  OOM in Trial {trial.number}, Fold {fold_idx + 1}! Skipping trial.")
            _cleanup_cuda()
            if os.path.exists(fold_output_dir):
                shutil.rmtree(fold_output_dir, ignore_errors=True)
            raise optuna.TrialPruned()

        finally:
            for var in ["trainer", "model", "training_args", "train_ds", "val_ds"]:
                try:
                    exec(f"del {var}")
                except NameError:
                    pass
            _cleanup_cuda()
            if os.path.exists(fold_output_dir):
                shutil.rmtree(fold_output_dir, ignore_errors=True)

        # Report intermediate result and check for pruning
        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            print(f"  Trial {trial.number} PRUNED after Fold {fold_idx + 1}")
            raise optuna.TrialPruned()

    hu.store_trial_summary(trial, N_FOLDS, ALL_LABELS)

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    print(f"\n  -> Trial {trial.number}: F1 Macro = {mean_f1:.4f} +/- {std_f1:.4f}")

    return mean_f1


print("Objective function defined.")
print(f"Folds: {N_FOLDS}")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")

## Start / Resume Study (Chunk Mode)
This cell is the core of the chunk mode:
- With `"new"`: creates a new DB + study, runs N_TRIALS_THIS_RUN trials
- With `"continue"`: loads an existing DB, runs N_TRIALS_THIS_RUN **additional** trials

In [ ]:
# ===== OPTUNA STUDY — CHUNK MODE =====
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from pathlib import Path
import glob

# --- Determine DB path ---
DB_BASE_DIR = Path(pu.OPTUNA_DB_DIR)
LATEST_DB_FILE = DB_BASE_DIR / ".latest_hpt_db.txt"

if RUN_MODE == "new":
    # Create a new DB with auto-incrementing number
    existing_dbs = sorted(glob.glob(str(DB_BASE_DIR / f"{STUDY_NAME}_*.db")))
    if existing_dbs:
        # Find highest number and increment
        last_num = 0
        for db in existing_dbs:
            try:
                num = int(Path(db).stem.split("_")[-1])
                last_num = max(last_num, num)
            except ValueError:
                pass
        db_number = last_num + 1
    else:
        db_number = 1
    db_filename = f"{STUDY_NAME}_{db_number:03d}.db"
    storage_path = str(DB_BASE_DIR / db_filename)
    storage_url = f"sqlite:///{storage_path}"
    print(f"[NEW] Creating new DB: {storage_path}")

elif RUN_MODE == "continue":
    if DB_PATH_OVERRIDE:
        storage_path = DB_PATH_OVERRIDE
    elif LATEST_DB_FILE.exists():
        storage_path = LATEST_DB_FILE.read_text().strip()
    else:
        # Fallback: find most recent DB in directory
        existing_dbs = sorted(glob.glob(str(DB_BASE_DIR / f"{STUDY_NAME}_*.db")))
        if not existing_dbs:
            raise FileNotFoundError(
                f"No existing DB found in {DB_BASE_DIR}! "
                f"Please set RUN_MODE='new' or provide DB_PATH_OVERRIDE."
            )
        storage_path = existing_dbs[-1]
        print(f"[AUTO] Most recent DB found: {storage_path}")

    if not os.path.exists(storage_path):
        raise FileNotFoundError(f"DB not found: {storage_path}")

    storage_url = f"sqlite:///{storage_path}"
    print(f"[CONTINUE] Loading existing DB: {storage_path}")

else:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE}. Allowed values: 'new', 'continue'")

# Remember latest DB for the next "continue" run
LATEST_DB_FILE.write_text(storage_path)
print(f"  -> Saved as latest DB in: {LATEST_DB_FILE}")

# --- Sampler + Pruner ---
sampler = TPESampler(seed=OPTUNA_SEED, n_startup_trials=5)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)

# --- Create or load study ---
if RUN_MODE == "new":
    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=storage_url,
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=False,
    )
    print(f"New study created: {STUDY_NAME}")
else:
    study = optuna.load_study(
        study_name=STUDY_NAME,
        storage=storage_url,
        sampler=sampler,
        pruner=pruner,
    )
    n_existing = len(study.trials)
    n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"Study loaded: {n_existing} existing trials ({n_complete} complete, {n_pruned} pruned)")

# --- Status before run ---
n_before = len(study.trials)
print(f"\n{'='*60}")
print(f"Trials before this run: {n_before}")
print(f"Trials this run:        {N_TRIALS_THIS_RUN}")
print(f"{'='*60}")

# --- Run trials ---
timer_hpt = pu.ExperimentTimer()
with timer_hpt:
    study.optimize(
        objective,
        n_trials=N_TRIALS_THIS_RUN,
        timeout=OPTUNA_TIMEOUT,
        gc_after_trial=True,
    )

n_after = len(study.trials)
print(f"\nChunk complete: {timer_hpt.duration_formatted}")
print(f"Trials executed this run: {n_after - n_before}")
print(f"Total trials in DB: {n_after}")

# --- Summary ---
all_trials = study.trials
n_complete_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.COMPLETE])
n_pruned_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.PRUNED])
n_failed_total = len([t for t in all_trials if t.state == optuna.trial.TrialState.FAIL])

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"  Complete: {n_complete_total}")
print(f"  Pruned:   {n_pruned_total}")
print(f"  Failed:   {n_failed_total}")
print(f"  Total:    {n_after}")

if n_complete_total > 0:
    print(f"\n  Best trial:          {study.best_trial.number}")
    print(f"  Best F1 Macro:       {study.best_value:.4f}")
    print(f"\n  Best hyperparameters:")
    for key, val in study.best_params.items():
        print(f"    {key}: {val}")
else:
    print("\n  No completed trials yet — more runs needed.")

print(f"\nDB: {storage_path}")
print(f"\nNext step: set RUN_MODE='continue' and run again,")
print(f"or proceed with the analysis cells below.")
print(f"{'='*60}")

In [ ]:
# ===== OPTUNA QUICK SANITY CHECK =====
import optuna.visualization as vis

if n_complete_total >= 2:
    fig_history = vis.plot_optimization_history(study)
    fig_history.update_layout(title="Optimization History: F1 Macro across Trials")
    fig_history.show()

    try:
        fig_importance = vis.plot_param_importances(study)
        fig_importance.update_layout(title="Parameter Importance (fANOVA)")
        fig_importance.show()
    except Exception as e:
        print(f"Parameter importance not available: {e}")

    print("Sanity-check plots created.")
else:
    print(f"Only {n_complete_total} completed trial(s) — at least 2 needed for plots.")
    print("Run more trials with RUN_MODE='continue'.")

In [ ]:
# ===== QUICK RESULTS OVERVIEW =====
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))
trials_df = trials_df.sort_values("value", ascending=False)

completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
pruned = trials_df[trials_df["state"] == "PRUNED"].copy()

print(f"Trials: {len(completed)} completed, {len(pruned)} pruned")

if len(completed) > 0:
    print(f"\nTop 5 trials:")
    top5_cols = ["number", "value"] + [c for c in completed.columns if c.startswith("params_")]
    print(completed[top5_cols].head().to_string(index=False))

    print(f"\nStatistics across all completed trials:")
    print(f"  Mean F1:   {completed['value'].mean():.4f}")
    print(f"  Std F1:    {completed['value'].std():.4f}")
    print(f"  Min F1:    {completed['value'].min():.4f}")
    print(f"  Max F1:    {completed['value'].max():.4f}")
else:
    print("No completed trials yet.")

## Generate Report (optional)
Only run once the desired number of trials has been reached.

In [ ]:
# ===== GENERATE HPT REPORT =====
import json
from datetime import datetime

if n_complete_total == 0:
    print("No completed trials — report cannot be generated.")
else:
    now = datetime.now()
    report_name = f"{now.strftime('%d%m%y')}_{MODEL_SHORT_NAME}_hpt_phase1"

    duration_str = timer_hpt.duration_formatted if 'timer_hpt' in dir() else "N/A (loaded from DB)"

    # --- Markdown report ---
    report_lines = [
        f"# HPT Report: EuroBERT-210M Phase 1",
        f"**Generated:** {now.strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        "---",
        "",
        "## Configuration",
        "| Property | Value |",
        "|---|---|",
        f"| Model | {MODEL_ID} |",
        f"| Total Trials | {len(study.trials)} |",
        f"| N Folds | {N_FOLDS} |",
        f"| Completed Trials | {len(completed)} |",
        f"| Pruned Trials | {len(pruned)} |",
        f"| Last Run Duration | {duration_str} |",
        f"| GPU | {pu.get_gpu_info()['gpu_name']} |",
        f"| CV Pool Size | {len(cv_pool_df)} |",
        f"| Test Size (frozen) | {len(test_df)} |",
        f"| Effective Batch Size | {EFFECTIVE_BATCH_SIZE} |",
        f"| DB Path | {storage_path} |",
        "",
        "## Fixed Parameters",
        "| Parameter | Value |",
        "|---|---|",
        f"| bf16 | {FIXED_BF16} |",
        f"| fp16 | {FIXED_FP16} |",
        f"| gradient_checkpointing | {FIXED_GRADIENT_CHECKPOINTING} |",
        f"| group_by_length | {FIXED_GROUP_BY_LENGTH} |",
        f"| optim | {FIXED_OPTIM} |",
        f"| early_stopping_patience | {FIXED_EARLY_STOPPING_PATIENCE} |",
        f"| logging_steps | {FIXED_LOGGING_STEPS} |",
        f"| max_length | {MAX_LENGTH} |",
        "",
        "## Search Ranges",
        "| Parameter | Range |",
        "|---|---|",
    ]
    for param, range_val in HP_RANGES.items():
        report_lines.append(f"| {param} | {range_val} |")

    report_lines += [
        "",
        f"**Best F1 Macro (CV Mean): {study.best_value:.4f}**",
        f"**Best Trial: {study.best_trial.number}**",
        "",
        "## All Trials",
        "",
    ]

    table_cols = ["number", "value", "state"] + [c for c in trials_df.columns if c.startswith("params_")]
    report_lines.append(trials_df[table_cols].to_markdown(index=False))
    report_lines += [
        "",
        "---",
        "*Best parameter extraction: see analyse_hyperparameter_tuning.ipynb*",
        f"*Generated by 02_hyperparameter_optimization.ipynb (chunk mode)*",
    ]

    report_md = "\n".join(report_lines)
    report_path = Path(pu.REPORTS_DIR) / f"{report_name}.md"
    report_path.write_text(report_md, encoding="utf-8")

    # --- JSON sidecar ---
    json_data = {
        "report_id": report_name,
        "model_id": MODEL_ID,
        "total_trials": len(study.trials),
        "n_folds": N_FOLDS,
        "best_value": round(study.best_value, 4),
        "best_trial_number": study.best_trial.number,
        "best_params": study.best_params,
        "all_trials": [
            {
                "number": t.number,
                "value": round(t.value, 4) if t.value is not None else None,
                "params": t.params,
                "state": str(t.state.name),
                "duration_s": round(t.duration.total_seconds(), 1) if t.duration else None,
            }
            for t in study.trials
        ],
        "search_ranges": {k: str(v) for k, v in HP_RANGES.items()},
        "fixed_params": {
            "bf16": FIXED_BF16,
            "fp16": FIXED_FP16,
            "gradient_checkpointing": FIXED_GRADIENT_CHECKPOINTING,
            "optim": FIXED_OPTIM,
            "early_stopping_patience": FIXED_EARLY_STOPPING_PATIENCE,
            "group_by_length": FIXED_GROUP_BY_LENGTH,
            "max_length": MAX_LENGTH,
            "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        },
        "last_run_duration": duration_str,
        "gpu": pu.get_gpu_info(),
        "db_path": storage_path,
    }
    json_path = Path(pu.REPORTS_DIR) / f"{report_name}.json"
    json_path.write_text(json.dumps(json_data, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Report: {report_path}")
    print(f"JSON:   {json_path}")

In [ ]:
# ===== NOTE: PARAMETER EXTRACTION =====
print("Best parameter extraction: see analyse_hyperparameter_tuning.ipynb")
print(f"DB location: {storage_path}")

In [ ]:
# ===== CLEANUP + AUTO-SHUTDOWN =====
import shutil

if os.path.exists("/content/hpt_tmp"):
    for item in Path("/content/hpt_tmp").iterdir():
        if item.name != "tb_logs":
            shutil.rmtree(item, ignore_errors=True)
    print("Temporary checkpoint files deleted.")
    print("TensorBoard logs kept: /content/hpt_tmp/tb_logs/")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU VRAM free: {free_mem:.1f} GB")

print(f"\nResults on Google Drive:")
print(f"  DB: {storage_path}")
if 'report_path' in dir():
    print(f"  Report: {report_path}")
    print(f"  JSON:   {json_path}")

# Terminate runtime (Colab Pro — saves credits)
# Comment out if you want to run more chunks!
# print("\nTerminating runtime...")
# from google.colab import runtime
# runtime.unassign()
print("\nDone! Runtime stays active for additional runs.")
print("For next run: set RUN_MODE='continue' and re-execute cells.")